# PKLot Dataset
[View on Kaggle](https://www.kaggle.com/datasets/ammarnassanalhajali/pklot-dataset/data)

## About This Dataset
The PKLot dataset contains **12,416 parking lot images** extracted from surveillance camera frames.

### Key Highlights
- Captured under diverse weather conditions: **sunny**, **cloudy**, and **rainy**
- Each parking space is labeled as **Occupied** or **Empty**
- Original rotated-rectangle annotations were converted into standard **object detection formats** using bounding boxes

## License and Usage
This dataset is distributed under the **Creative Commons Attribution 4.0 (CC BY 4.0)** license.

If you use this dataset in research or publications, please acknowledge the source and cite the original PKLot paper.

## Recommended Citation
> Almeida, P., Oliveira, L. S., Silva Jr, E., Britto Jr, A., Koerich, A.
> **PKLot - A robust dataset for parking lot classification.**
> *Expert Systems with Applications*, 42(11):4937-4949, 2015.

## COCO to YOLO Conversion

The cells below convert the split-wise COCO files (`_annotations.coco.json`) into YOLO label files and generate a ready-to-train `data.yaml`.

In [1]:
from pathlib import Path
from collections import defaultdict, Counter
import json
import shutil


def resolve_pklot_root() -> Path:
    """Resolve the project root for common notebook working directories."""
    cwd = Path.cwd().resolve()
    project_name = "PKLot-Parking-Occupancy-Detection"
    candidates = [
        cwd,
        cwd / project_name,
        cwd.parent / project_name,
        cwd.parent,
    ]

    unique_candidates = list(dict.fromkeys(candidates))
    for candidate in unique_candidates:
        if (candidate / "Dataset" / "train" / "_annotations.coco.json").exists():
            return candidate

    raise FileNotFoundError(
        "Could not find PKLot dataset root. Checked: "
        + ", ".join(str(path) for path in unique_candidates)
        + " (expected Dataset/train/_annotations.coco.json)"
    )


PKLOT_ROOT = resolve_pklot_root()
DATASET_ROOT = PKLOT_ROOT / "Dataset"
YOLO_ROOT = PKLOT_ROOT / "Dataset_yolo"
SPLIT_MAP = {"train": "train", "valid": "val", "test": "test"}

print(f"PKLOT_ROOT: {PKLOT_ROOT}")
print(f"DATASET_ROOT: {DATASET_ROOT}")
print(f"YOLO_ROOT: {YOLO_ROOT}")

for split in SPLIT_MAP:
    split_dir = DATASET_ROOT / split
    coco_file = split_dir / "_annotations.coco.json"
    print(f"{split:>5}: split_dir_exists={split_dir.exists()} | coco_exists={coco_file.exists()}")

PKLOT_ROOT: D:\Projects--Kaggle-\PKLot-Parking-Occupancy-Detection
DATASET_ROOT: D:\Projects--Kaggle-\PKLot-Parking-Occupancy-Detection\Dataset
YOLO_ROOT: D:\Projects--Kaggle-\PKLot-Parking-Occupancy-Detection\Dataset_yolo
train: split_dir_exists=True | coco_exists=True
valid: split_dir_exists=True | coco_exists=True
 test: split_dir_exists=True | coco_exists=True


In [2]:
def clamp01(value: float) -> float:
    return max(0.0, min(1.0, value))


def link_or_copy_image(src: Path, dst: Path) -> None:
    """Use hard-links when possible to avoid duplicating image storage."""
    if dst.exists():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        dst.hardlink_to(src)
    except Exception:
        shutil.copy2(src, dst)


def coco_bbox_to_yolo(bbox, img_w: int, img_h: int):
    x, y, w, h = bbox
    if img_w <= 0 or img_h <= 0 or w <= 0 or h <= 0:
        return None

    x_center = clamp01((x + (w / 2.0)) / img_w)
    y_center = clamp01((y + (h / 2.0)) / img_h)
    width = clamp01(w / img_w)
    height = clamp01(h / img_h)

    if width <= 0.0 or height <= 0.0:
        return None
    return x_center, y_center, width, height


def convert_split_to_yolo(
    split_name: str,
    yolo_split_name: str,
    category_id_to_yolo: dict,
) -> dict:
    split_dir = DATASET_ROOT / split_name
    coco_path = split_dir / "_annotations.coco.json"

    with coco_path.open("r", encoding="utf-8") as f:
        coco = json.load(f)

    images_dir = YOLO_ROOT / "images" / yolo_split_name
    labels_dir = YOLO_ROOT / "labels" / yolo_split_name
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    image_records = {img["id"]: img for img in coco.get("images", [])}
    anns_by_image = defaultdict(list)
    for ann in coco.get("annotations", []):
        anns_by_image[ann.get("image_id")].append(ann)

    class_counter = Counter()
    missing_images = 0
    skipped_annotations = 0

    for image_id, img in image_records.items():
        file_name = img.get("file_name")
        if not file_name:
            continue

        src_img = split_dir / file_name
        dst_img = images_dir / file_name

        if not src_img.exists():
            missing_images += 1
            continue

        link_or_copy_image(src_img, dst_img)

        label_path = labels_dir / f"{Path(file_name).stem}.txt"
        lines = []

        for ann in anns_by_image.get(image_id, []):
            coco_cat_id = ann.get("category_id")
            if coco_cat_id not in category_id_to_yolo:
                skipped_annotations += 1
                continue

            bbox = ann.get("bbox")
            if not bbox or len(bbox) != 4:
                skipped_annotations += 1
                continue

            converted = coco_bbox_to_yolo(bbox, int(img.get("width", 0)), int(img.get("height", 0)))
            if converted is None:
                skipped_annotations += 1
                continue

            yolo_cat_id = category_id_to_yolo[coco_cat_id]
            x_center, y_center, width, height = converted
            lines.append(f"{yolo_cat_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
            class_counter[yolo_cat_id] += 1

        label_path.write_text("\n".join(lines), encoding="utf-8")

    return {
        "split": split_name,
        "yolo_split": yolo_split_name,
        "images": len(image_records),
        "labels_written": sum(1 for _ in labels_dir.glob("*.txt")),
        "missing_images": missing_images,
        "skipped_annotations": skipped_annotations,
        "class_counts": dict(class_counter),
    }

In [3]:
# Read categories from train split and map COCO category ids to contiguous YOLO ids.
with (DATASET_ROOT / "train" / "_annotations.coco.json").open("r", encoding="utf-8") as f:
    train_coco = json.load(f)

categories = sorted(train_coco.get("categories", []), key=lambda c: c["id"])
category_id_to_yolo = {cat["id"]: idx for idx, cat in enumerate(categories)}
yolo_names = [cat["name"] for cat in categories]

print("Class mapping (COCO -> YOLO):")
for cat in categories:
    print(f"  {cat['id']} -> {category_id_to_yolo[cat['id']]} | {cat['name']}")

# Prepare output root.
if YOLO_ROOT.exists():
    shutil.rmtree(YOLO_ROOT)
YOLO_ROOT.mkdir(parents=True, exist_ok=True)

summaries = []
for coco_split, yolo_split in SPLIT_MAP.items():
    summary = convert_split_to_yolo(coco_split, yolo_split, category_id_to_yolo)
    summaries.append(summary)

# Write Ultralytics-compatible data.yaml.
names_yaml = "\n".join([f"  {idx}: {name}" for idx, name in enumerate(yolo_names)])
data_yaml_text = (
    f"path: {YOLO_ROOT.as_posix()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "names:\n"
    f"{names_yaml}\n"
)
(DATASET_ROOT.parent / "Dataset_yolo" / "data.yaml").write_text(data_yaml_text, encoding="utf-8")

print("\nConversion summary:")
for s in summaries:
    print(
        f"- {s['split']} -> {s['yolo_split']} | images={s['images']} | "
        f"label_files={s['labels_written']} | missing_images={s['missing_images']} | "
        f"skipped_annotations={s['skipped_annotations']}"
    )

print("\nClass counts by YOLO id:")
total_counts = Counter()
for s in summaries:
    total_counts.update(s["class_counts"])
for class_id in sorted(total_counts):
    print(f"  {class_id} ({yolo_names[class_id]}): {total_counts[class_id]}")

print(f"\nWrote data config: {YOLO_ROOT / 'data.yaml'}")

Class mapping (COCO -> YOLO):
  0 -> 0 | spaces
  1 -> 1 | space-empty
  2 -> 2 | space-occupied

Conversion summary:
- train -> train | images=8691 | label_files=8691 | missing_images=0 | skipped_annotations=0
- valid -> val | images=2483 | label_files=2483 | missing_images=0 | skipped_annotations=0
- test -> test | images=1242 | label_files=1242 | missing_images=0 | skipped_annotations=0

Class counts by YOLO id:
  1 (space-empty): 376121
  2 (space-occupied): 335735

Wrote data config: D:\Projects--Kaggle-\PKLot-Parking-Occupancy-Detection\Dataset_yolo\data.yaml


In [4]:
# Quick integrity checks after conversion.
for split in ["train", "val", "test"]:
    image_dir = YOLO_ROOT / "images" / split
    label_dir = YOLO_ROOT / "labels" / split

    image_count = sum(1 for p in image_dir.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})
    label_count = sum(1 for p in label_dir.iterdir() if p.suffix.lower() == ".txt")

    print(f"{split:>5}: images={image_count}, labels={label_count}")

# Show a sample label file preview.
sample_label = next((YOLO_ROOT / "labels" / "train").glob("*.txt"), None)
if sample_label is not None:
    print(f"\nSample label file: {sample_label.name}")
    print("-" * 40)
    print(sample_label.read_text(encoding="utf-8")[:500])
else:
    print("No training label files found.")

train: images=8691, labels=8691
  val: images=2483, labels=2483
 test: images=1242, labels=1242

Sample label file: 2012-09-11_15_16_58_jpg.rf.61d961a86c9a16694403dfcb72cd450c.txt
----------------------------------------
2 0.235156 0.289062 0.035937 0.062500
1 0.259766 0.291016 0.035156 0.066406
2 0.285547 0.291016 0.033594 0.066406
1 0.311719 0.289453 0.035937 0.066406
2 0.335547 0.290625 0.033594 0.062500
2 0.377344 0.288281 0.032813 0.064062
1 0.404297 0.289062 0.027344 0.059375
1 0.430078 0.291406 0.025781 0.064062
1 0.455469 0.288281 0.026562 0.064062
1 0.481250 0.289844 0.028125 0.060937
1 0.506250 0.286328 0.025000 0.053906
2 0.533594 0.286328 0.026562 0.057031
2 0.560937 0.282813 0.025000 0.053125
1 0.58
